# Chapter 16: Automated Machine Learning (AutoML)

**Goal**  
Learn low-code / no-code tools that automatically select algorithms, tune hyperparameters, and build strong models — perfect when time is limited or the analyst is not an ML specialist.

### Libraries Covered
| Library | Style | Best for |
|---------|-------|----------|
| **PyCaret** | Low-code, very beginner-friendly | Fast experimentation & comparison |
| **FLAML** | Lightweight & fast | Resource-constrained environments |
| **AutoGluon** | High accuracy, multi-modal | When you want state-of-the-art results |
| **Auto-sklearn** | Full AutoML on scikit-learn | Academic / research style |
| **H2O AutoML** | Distributed, enterprise | Large data & production teams |

---

## 16.1 Why AutoML for MBA Analysts?

- You can go from raw data → strong model in **minutes** instead of days.
- Great for quick prototypes, POCs, and baseline models.
- Still requires business understanding to interpret results and avoid leakage.

> AutoML is a force multiplier, not a replacement for domain knowledge.

In [ ]:
# Note: Some AutoML libraries are heavy. Install only what you need.
# !pip install pycaret flaml autogluon -q

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

np.random.seed(42)
print("Ready")

### Prepare a simple churn dataset (same style as previous chapters)

In [ ]:
n = 3000
df = pd.DataFrame({
    "tenure": np.random.randint(1, 72, n),
    "monthly_charges": np.round(np.random.uniform(20, 120), 2),
    "contract": np.random.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.2]),
    "internet": np.random.choice(["DSL", "Fiber", "None"], n),
    "tech_support": np.random.choice(["Yes", "No"], n, p=[0.3, 0.7]),
    "senior": np.random.binomial(1, 0.16, n),
    "tickets": np.random.poisson(1.3, n)
})
logit = (-1.7 + 0.018*df.monthly_charges - 0.03*df.tenure
         + 0.85*(df.contract=="Month-to-month").astype(int)
         + 0.4*(df.tech_support=="No").astype(int) + 0.3*df.tickets + 0.35*df.senior)
df["churn"] = (np.random.rand(n) < 1/(1+np.exp(-logit))).astype(int)

print(df.churn.value_counts(normalize=True).round(3))
df.head()

---

## 16.2 PyCaret – The Fastest Way to Experiment

PyCaret wraps dozens of models behind a simple API.

In [ ]:
# PyCaret example (commented so the notebook runs without heavy install)
pycaret_code = '''
from pycaret.classification import *

# Setup (handles preprocessing, train/test split, etc.)
s = setup(data=df, target="churn", session_id=42,
          categorical_features=["contract", "internet", "tech_support"],
          silent=True, verbose=False)

# Compare many models
best = compare_models(include=["lr", "rf", "xgboost", "lightgbm", "catboost"], sort="AUC")

# Finalize and predict
final = finalize_model(best)
predictions = predict_model(final)
'''
print("PyCaret workflow (copy & run after installing pycaret):")
print(pycaret_code)

---

## 16.3 FLAML – Lightweight & Fast AutoML

In [ ]:
flaml_code = '''
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    metric="roc_auc",
    time_budget=60,          # seconds
    estimator_list=["lgbm", "xgboost", "rf", "extra_tree"],
    verbose=0
)

print("Best model:", automl.best_estimator)
print("Best config:", automl.best_config)
print("Best AUC:", automl.best_loss)   # actually 1 - AUC for some metrics
'''
print("FLAML example:")
print(flaml_code)

---

## 16.4 AutoGluon – High Accuracy with Stack Ensembling

AutoGluon is known for producing very strong models with almost no effort, especially on tabular data.

In [ ]:
autogluon_code = '''
from autogluon.tabular import TabularPredictor

train_data = df.copy()
predictor = TabularPredictor(label="churn", eval_metric="roc_auc").fit(
    train_data,
    time_limit=120,          # seconds
    presets="medium_quality" # or "best_quality" for longer runs
)

leaderboard = predictor.leaderboard(silent=True)
print(leaderboard.head())

# Predict on new data
# preds = predictor.predict_proba(test_data)
'''
print("AutoGluon example:")
print(autogluon_code)

---

## 16.5 Auto-sklearn & H2O (Overview)

**Auto-sklearn**  
- Uses Bayesian optimization + meta-learning.  
- Drop-in replacement for a scikit-learn estimator.  
- Heavier dependency footprint.

**H2O AutoML**  
- Excellent for larger datasets.  
- Runs many algorithms in parallel (including stacked ensembles).  
- Has a nice web UI and is popular in enterprise settings.

Both are powerful; start with PyCaret or AutoGluon for most MBA use-cases.

---

## Business Challenge – Chapter 16

1. Install one AutoML library (PyCaret **or** FLAML **or** AutoGluon).
2. Run it on the churn dataset (or any dataset of your choice).
3. Report the best model and its ROC-AUC / accuracy.
4. Extract feature importance or SHAP values if available and write two business recommendations.

Document your findings in the cell below.

In [ ]:
# Your AutoML experiment notes & results


---

## Key Takeaways
- AutoML dramatically shortens the time from data to usable model.
- **PyCaret** is the friendliest for beginners and rapid comparison.
- **AutoGluon** often delivers the highest accuracy with minimal tuning.
- **FLAML** is ideal when compute time or memory is limited.
- Always validate results with business logic and hold-out data — AutoML can still overfit or leak.

**Next**: Deep Learning foundations for business applications